# Task 5 — Duplicates and the distinct count

"Remove duplicates" is not an instruction until somebody says what a duplicate
is. Four definitions, applied to the same **1,260** loadable staging rows.
Three of them are wrong, and they are wrong in two different directions: two
keep rows they should drop, one **drops rows it should keep**.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True
pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows


  rail_march.xlsx: 315 rows
  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)


registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


In [5]:
# The four definitions on the same 1,260 loadable rows.
loadable["_d"] = loadable["ShipDate"].map(parse_date)
loadable["_o"] = loadable["Origin"].map(lambda v: (v or "").strip())
loadable["_t"] = loadable["Destination"].map(lambda v: (v or "").strip())
loadable["_wkg"] = loadable.apply(lambda r: plain_number(r["WeightKG"]) * (1000 if "rail" in r["SourceFile"].lower() else 1), axis=1)
d1 = loadable.drop_duplicates(subset=CONTRACT, keep="first")
d2 = loadable.drop_duplicates(subset=["ShipmentID"], keep="first")
d3 = loadable.drop_duplicates(subset=["_n"], keep="first")
d4 = loadable.drop_duplicates(subset=["_d", "_o", "_t", "_wkg", "CustomerID"], keep="first")
defs = pd.DataFrame([
    (1, "the whole row is identical, every column compared", len(d1)),
    (2, "ShipmentID is identical, compared as it arrived", len(d2)),
    (3, "ShipmentID is identical, compared AFTER trim and upper case", len(d3)),
    (4, "same day, same lane, same weight, same customer", len(d4)),
], columns=["Definition", "What counts as a duplicate", "Rows surviving"])
print(defs.to_string(index=False))
print(f"\nloadable rows: {len(loadable)}; definition 3 (the grain) removes {len(loadable) - len(d3)} second copies")

# Definition 4 is the dangerous one, because it sounds more rigorous than the
# right answer. The groups it destroys:
grp = loadable.groupby(["_d", "_o", "_t", "_wkg", "CustomerID"])["_n"].nunique()
for key in grp[grp > 1].index:
    sub = loadable[(loadable["_d"] == key[0]) & (loadable["_o"] == key[1]) & (loadable["_t"] == key[2]) & (loadable["_wkg"] == key[3]) & (loadable["CustomerID"] == key[4])]
    print(f"  D4 destroys: {sorted(sub['_n'].unique())} — {key[0]} {key[1]}->{key[2]} {key[3]:,.0f} kg {key[4]}")

 Definition                                  What counts as a duplicate  Rows surviving
          1           the whole row is identical, every column compared            1212
          2             ShipmentID is identical, compared as it arrived            1212
          3 ShipmentID is identical, compared AFTER trim and upper case            1200
          4             same day, same lane, same weight, same customer            1198

loadable rows: 1260; definition 3 (the grain) removes 60 second copies
  D4 destroys: ['RD-5216', 'RD-5337'] — 2026-03-15 Lahore->Sialkot 6,300 kg CU-136
  D4 destroys: ['RD-5332', 'RD-5409'] — 2026-03-29 Karachi->Hyderabad 6,500 kg CU-102


In [6]:
# The customer count, four ways.
reg_n = len(cust)
fact_ids = deduped["CustomerID"].nunique()  # original IDs on the 1,200 loaded rows, before conform
businesses = len(dim_customer) - 1
dim_rows_total = len(dim_customer)
print(f"registry rows: {reg_n}")
print(f"distinct CustomerID in the fact table (original IDs): {fact_ids} — every registry ID was used in March, reproducing the registry's mistake")
print(f"distinct businesses after conforming names: {businesses}")
print(f"DimCustomer rows incl. Unknown: {dim_rows_total} (40 businesses + -1)")
print("\nthe 6 businesses typed twice (loser -> survivor):")
for loser, surv in sorted(alias_map.items()):
    print(f"  CU-{loser} -> CU-{surv}")

registry rows: 46
distinct CustomerID in the fact table (original IDs): 46 — every registry ID was used in March, reproducing the registry's mistake
distinct businesses after conforming names: 40
DimCustomer rows incl. Unknown: 41 (40 businesses + -1)

the 6 businesses typed twice (loser -> survivor):
  CU-901 -> CU-101
  CU-902 -> CU-103
  CU-903 -> CU-104
  CU-904 -> CU-109
  CU-905 -> CU-119
  CU-906 -> CU-131


In [7]:
assert len(loadable) == 1260
assert (len(d1), len(d2), len(d3), len(d4)) == (1212, 1212, 1200, 1198)
assert len(grp[grp > 1]) == 2
assert (reg_n, fact_ids, businesses, dim_rows_total) == (46, 46, 40, 41)
assert len(alias_map) == 6
print("SELF-CHECK PASS — 1212 / 1212 / 1200 / 1198; customers 46 / 46 / 40 / 41")

# Analysis
# The duplication lives in the DIMENSION (registry), not the fact: 6
# businesses typed twice (CU-101/901 Indus Textiles; CU-103/902 Karachi Steel;
# CU-104/903 Pak Agro; CU-109/904 Sialkot Sports; CU-119/905 Sukkur Rice;
# CU-131/906 Lahore Tyre). Fix: conform — merge each pair into one
# DimCustomer row, keep the losing spelling as an alias, point fact
# CustomerKeys at the surviving business. Totals do not move because
# conforming changes KEYS, not rows or measures: no fact row is added or
# removed and no weight or cost changes — only the customer count drops
# 46 -> 40, because a count over a dimension is not a sum over facts.

SELF-CHECK PASS — 1212 / 1212 / 1200 / 1198; customers 46 / 46 / 40 / 41
